# 在 Google Colab 上訓練 DQN 自動駕駛

訓練完全不需要畫面（不用 pygame、不會開視窗），只需要 numpy 和 PyTorch，Colab 都已經內建。

1. **執行階段 → 變更執行階段類型 → T4 GPU**
2. 把 `vertical-shmup.zip` 上傳到 Google Drive 根目錄（`MyDrive/`）。
3. 依序執行每個 cell。存檔會寫到 `MyDrive/shmup/`，斷線也不會不見。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/shmup'
!mkdir -p $DRIVE

In [ ]:
!rm -rf /content/vertical-shmup
!unzip -q -o /content/drive/MyDrive/vertical-shmup.zip -d /content
%cd /content/vertical-shmup
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import torch, numpy; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

## 訓練
log 會存到 `MyDrive/shmup/train.log`。第一次建議先跑 `--total-steps 3000000`（約 20–35 分鐘），預設的 GPU 設定是 800 萬步。

In [ ]:
!python -u scripts/train_gpu.py --total-steps 3000000 --save-path $DRIVE/dqn.pt 2>&1 | tee -a $DRIVE/train.log

## 斷線後續訓
重新執行前兩個 cell，再執行這一個，就會從最後一次評估的存檔接著訓練。

In [ ]:
!python -u scripts/train_gpu.py --total-steps 3000000 --save-path $DRIVE/dqn.pt --resume $DRIVE/dqn_last.pt 2>&1 | tee -a $DRIVE/train.log

## 評估並匯出網頁版模型

In [ ]:
!python scripts/train_gpu.py --eval-only --save-path $DRIVE/dqn.pt
!python scripts/export_web.py --model $DRIVE/dqn.pt
!cp web/model/dqn_model.js $DRIVE/

In [ ]:
from google.colab import files
files.download(DRIVE + '/dqn.pt')
files.download(DRIVE + '/dqn_model.js')